# NLP ASSIGNMENT-5: # Fine-Tuning DistilBERT for POS Tagging & Chunking

In [1]:
!pip install --upgrade pip
!pip install "transformers>=4.41.0" --only-binary=:all:
!pip install "tokenizers>=0.19.1" --only-binary=:all:

  Using cached pip-26.0.1-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-26.0.1-py3-none-any.whl (1.8 MB)



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: To modify pip, please run the following command:
C:\Users\user\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

print("Tokenizer working ✅")

C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokenizer working ✅


## Dataset

In [3]:
from datasets import load_dataset

dataset = load_dataset("conll2003")
print(dataset)

C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\datasets\load.py:1486: FutureWarning: The repository for conll2003 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/conll2003
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


##  Tokenization

The dataset is tokenized using DistilBERT tokenizer.
Labels are aligned with tokens using word-level mapping.

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_and_align_labels(example):
    tokenized_input = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )
    
    labels = []
    word_ids = tokenized_input.word_ids()

    previous_word = None
    for word_id in word_ids:
        if word_id is None:
            labels.append(-100)
        elif word_id != previous_word:
            labels.append(example["ner_tags"][word_id])
        else:
            labels.append(-100)
        previous_word = word_id

    tokenized_input["labels"] = labels
    return tokenized_input


tokenized_dataset = dataset.map(tokenize_and_align_labels)

print("Tokenization done ✅")

Map: 100%|████████████████████████████████████████████████████████████████| 3250/3250 [00:02<00:00, 1197.07 examples/s]


Tokenization done ✅


## Model
DistilBERT for token classification.
It is a lightweight transformer model suitable for NLP tasks.

In [5]:
from transformers import AutoModelForTokenClassification

num_labels = len(dataset["train"].features["ner_tags"].feature.names)

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)

print("Model loaded ✅")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 2598.49it/s]
DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded ✅


In [11]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

## Training

The model is trained on a subset of the dataset for faster execution.
Training is done using Hugging Face Trainer API.

In [12]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,   # keep 1 for fast run
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="no"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"].select(range(100)),
    eval_dataset=tokenized_dataset["validation"].select(range(50)),
    data_collator=data_collator
)

print("Trainer ready ✅")

Trainer ready ✅


In [13]:
trainer.train()

print("Training complete ✅")

Step,Training Loss
10,1.806927


Training complete ✅


In [14]:
import torch

label_names = dataset["train"].features["ner_tags"].feature.names

# sample sentence
sentence = ["John", "works", "at", "Google"]

# tokenize
inputs = tokenizer(
    sentence,
    return_tensors="pt",
    is_split_into_words=True
)

# predict
with torch.no_grad():
    outputs = model(**inputs)

predictions = torch.argmax(outputs.logits, dim=2)

# map predictions
predicted_labels = [label_names[p.item()] for p in predictions[0]]

print("Word  →  Predicted Label")
for word, label in zip(sentence, predicted_labels):
    print(f"{word} → {label}")

Word  →  Predicted Label
John → B-ORG
works → O
at → O
Google → O


##  Conclusion

This project successfully demonstrates token classification using transformers.
The model can identify entities like persons, organizations, and locations.